# AIRBNB PRICE PREDICTION

### Project Overview
This project builds an end-to-end machine learning pipeline to predict Airbnb nightly rental prices from listing, location, amenity, and host features.

| Project Attribute | Details |
|---|---|
| **Project Title** | Airbnb Nightly Price Prediction System |
| **Academic Domain** | Applied Machine Learning and Predictive Real Estate Analytics |
| **Dataset Scope** | 74,111 listing records across six major U.S. metropolitan markets: NYC, LA, SF, DC, Chicago, and Boston |
| **Target Variable** | `log_price` — the natural logarithm of nightly price in USD |
| **Why Log Transform?** | Rental prices are right-skewed. Modeling the log target reduces the influence of extreme values and makes errors more proportional to price. |
| **Candidate Algorithms** | Median baseline, Ridge Regression, Decision Tree, Random Forest, Gradient Boosting, and HistGradientBoosting |
| **Evaluation Metrics** | R², RMSE, MAE, median percentage error, and prediction error within selected tolerances |

### Core Project Objectives
1. **Understand the data:** Inspect schema, distributions, and missing values.
2. **Clean the data:** Remove exact duplicates, handle missing values, and filter invalid target prices.
3. **Explore listing prices:** Compare prices by room type, capacity, and city.
4. **Engineer features:** Create capacity ratios, amenity indicators, host signals, and city-center distances.
5. **Prevent data leakage:** Keep imputation, scaling, and encoding inside scikit-learn pipelines.
6. **Compare models:** Benchmark regression algorithms with three-fold cross-validation.
7. **Tune the selected model:** Use randomized search to optimize HistGradientBoosting.
8. **Explain predictions:** Review feature importance, run what-if comparisons, and estimate prices for sample listings.

## 02 - Environment Setup

In [ ]:
import os
import re
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (GradientBoostingRegressor, HistGradientBoostingRegressor,RandomForestRegressor,)
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (KFold,RandomizedSearchCV,cross_val_score,train_test_split,)


RANDOM_STATE = 42
import warnings
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.autolayout": True,"font.size": 10,})
sns.set_theme(style="whitegrid")

## 03 - Dataset Download / Import / Load

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

data_file = Path("train.csv")
if data_file.exists():
    print(f"Local file found: {data_file.resolve()}")
else:
    print("Local train.csv not found. Downloading from Kaggle...")
    try:
        import kagglehub
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub
    dataset_dir = Path(kagglehub.dataset_download("stevezhenghp/airbnb-price-prediction"))
    matching_files = list(dataset_dir.rglob("train.csv"))
    if not matching_files:
        raise FileNotFoundError(f"Downloaded dataset has no train.csv: {dataset_dir}")
    data_file = matching_files[0]
    print(f"Downloaded file: {data_file}")

df = pd.read_csv(data_file)
print(f"Dataset dimensions: {df.shape[0]:,} rows × {df.shape[1]} columns")

# 04 - Dataset overview

In [ ]:
print("Dataset information")
df.info()

print("\nFirst five rows")
display(df.head())

print("\nLast five rows")
display(df.tail())

print("\nDescriptive statistics")
display(df.describe(include="all"))

In [ ]:
# Separate numerical and categorical features

num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = df.select_dtypes(
    include=["object", "category", "bool", "string"]).columns.tolist()
print(f"Total features: {df.shape[1]}\n")
print(f"Numerical features ({len(num_cols)}):")
print(num_cols)
print(f"\nCategorical features ({len(cat_cols)}):")
print(cat_cols)

In [ ]:
numeric_summary = df[["accommodates", "bathrooms", "bedrooms", "beds","number_of_reviews", "review_scores_rating"]].describe().round(2)
numeric_summary

In [ ]:
# Listing distributions across cities and property types

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
city_listing_counts = df["city"].value_counts()
sns.barplot(x=city_listing_counts.index,y=city_listing_counts.values, ax=axes[0], palette="Blues_r",)
axes[0].set_title("Number of Listings by City")
axes[0].set_xlabel("City")
axes[0].set_ylabel("Listing Count")

for bar in axes[0].patches:
    axes[0].annotate(
        f"{int(bar.get_height()):,}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center",
        va="bottom",
        xytext=(0, 3),
        textcoords="offset points",
    )


top_property_types = df["property_type"].value_counts().head(6)

sns.barplot(
    x=top_property_types.values,
    y=top_property_types.index,
    ax=axes[1],
    palette="Purples_r",
)
axes[1].set_title("Top 6 Property Types")
axes[1].set_xlabel("Listing Count")
axes[1].set_ylabel("Property Type")

plt.tight_layout()
plt.show()

In [ ]:
# Missing values by column

missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2),
})

missing_summary = (
    missing_summary[missing_summary["Missing Count"] > 0]
    .sort_values("Missing Percentage", ascending=False)
)

if missing_summary.empty:
    print("No missing values found.")
else:
    display(missing_summary)

## 06 - Data Cleaning

### Cleaning Strategy & Decisions
* **Dropping Non-Informative Columns:** `id`, `name`, `description`, `thumbnail_url`, and `zipcode` are removed. Record IDs risk memorization; unstructured free text requires heavy NLP without substantial predictive gain for tabular pricing; `zipcode` is high-cardinality and noisy, whereas `latitude` and `longitude` provide clean spatial coordinates.
* **Target Conversion & Outlier Filtering:** Compute `price = exp(log_price)`. Records with price below 10 USD or missing targets are filtered as invalid data-entry anomalies. High-end listings are retained because the log transform naturally balances their leverage.
* **Deduplication:** Remove exact duplicate rows to prevent data leakage and skewed evaluations.


In [ ]:
# Data cleaning
rows_before = len(df)
duplicate_rows = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
df["log_price"] = pd.to_numeric(df["log_price"], errors="coerce")
missing_target_rows = int(df["log_price"].isna().sum())
df = df.dropna(subset=["log_price"]).copy()
with np.errstate(over="ignore"):
    df["price"] = np.exp(df["log_price"])
invalid_price_mask = ~np.isfinite(df["price"]) | (df["price"] < 10)
invalid_price_rows = int(invalid_price_mask.sum())
df = df.loc[~invalid_price_mask].reset_index(drop=True)
columns_to_drop = ["id", "name", "description", "thumbnail_url", "zipcode"]
dropped_columns = [column for column in columns_to_drop if column in df.columns]
df = df.drop(columns=dropped_columns)
rows_removed = rows_before - len(df)
print(f"Rows before cleaning: {rows_before:,}")
print(f"Exact duplicate rows removed: {duplicate_rows:,}")
print(f"Rows with missing/invalid log_price removed: {missing_target_rows:,}")
print(f"Rows with non-finite or under-$10 prices removed: {invalid_price_rows:,}")
print(f"Total rows removed: {rows_removed:,}")
print(f"Rows after cleaning: {len(df):,}")
print(f"Columns removed: {len(dropped_columns)} ({', '.join(dropped_columns)})")
print(f"Columns remaining: {df.shape[1]:,}")

In [ ]:
# Price distribution summary after cleaning (USD)
print(f"Cleaned Price (USD):")
print(f"  Min:    ${df['price'].min():.2f}")
print(f"  25%:    ${df['price'].quantile(0.25):.2f}")
print(f"  Median: ${df['price'].median():.2f}")
print(f"  75%:    ${df['price'].quantile(0.75):.2f}")
print(f"  Mean:   ${df['price'].mean():.2f}")
print(f"  Max:    ${df['price'].max():.2f}")

## 07 - Exploratory Data Analysis (EDA)

In [ ]:
# Compare raw price and log-price distributions

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

price_99th_percentile = df["price"].quantile(0.99)
prices_below_99th_percentile = df.loc[df["price"] <= price_99th_percentile,"price",]

sns.histplot(prices_below_99th_percentile,bins=40,kde=True,ax=axes[0],color="#4C72B0",)
axes[0].set_title("Nightly Price Distribution (Up to 99th Percentile)")
axes[0].set_xlabel("Nightly Price (USD)")
axes[0].set_ylabel("Number of Listings")

sns.histplot( df["log_price"].dropna(), bins=40, kde=True, ax=axes[1], color="#55A868",)
axes[1].set_title("Log-Price Distribution")
axes[1].set_xlabel("Log Price")
axes[1].set_ylabel("Number of Listings")

plt.tight_layout()
plt.show()

In [ ]:
# Price distribution by room type
room_type_order = [ "Entire home/apt", "Private room","Shared room",]
price_99th_percentile = df["price"].quantile(0.99)
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Raw night price, capped for readability
sns.boxplot(data=df, x="room_type", y="price", order=room_type_order, ax=axes[0],palette="Set2",)
axes[0].set_ylim(0, price_99th_percentile)
axes[0].set_title(f"Nightly Price by Room Type (up to ${price_99th_percentile:,.0f})")
axes[0].set_xlabel("Room Type")
axes[0].set_ylabel("Nightly Price (USD)")

# Log-transformed nightly price
sns.boxplot(data=df,x="room_type",y="log_price",order=room_type_order,ax=axes[1],palette="Set2",)
axes[1].set_title("Log Price by Room Type")
axes[1].set_xlabel("Room Type")
axes[1].set_ylabel("Log Price")

plt.tight_layout()
plt.show()

In [ ]:
# Price vs. capacity features
price_cap = df["price"].quantile(0.99)
features = [
    ("accommodates", 10, "Number of Guests", "#4C72B0"),
    ("bedrooms", 6, "Bedrooms", "#55A868"),
    ("bathrooms", 4, "Bathrooms", "#C44E52"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
for ax, (column, max_value, label, color) in zip(axes, features):
    subset = df[df[column] <= max_value]
    sns.boxplot(data=subset, x=column, y="price", ax=ax, color=color)
    ax.set_ylim(0, price_cap)
    ax.set(title=f"Price vs. {label}", xlabel=label, ylabel="Nightly Price (USD)")

plt.tight_layout()
plt.show()

In [ ]:
# Price comparison across cities
price_cap = df["price"].quantile(0.99)
city_order = df.groupby("city")["price"].median().sort_values(ascending=False).index

fig, axes = plt.subplots(1, 2, figsize=(16, 5.2))

sns.boxplot(data=df, x="city", y="price", order=city_order, ax=axes[0], palette="Blues_r")
axes[0].set(ylim=(0, price_cap), title=f"Nightly Price by City (up to ${price_cap:,.0f})",
            xlabel="City", ylabel="Nightly Price (USD)")

sns.barplot(data=df, x="city", y="price", order=city_order, estimator=np.median,
            errorbar=None, ax=axes[1], palette="Blues_r")
axes[1].set(title="Median Nightly Price by City", xlabel="City", ylabel="Median Price (USD)")

for bar in axes[1].patches:
    axes[1].annotate(f"${bar.get_height():,.0f}",
                     (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                     ha="center", va="bottom", xytext=(0, 3), textcoords="offset points")

plt.tight_layout()
plt.show()

In [ ]:
# Median nightly price (USD) by city and room type
median_price_by_city_room = df.pivot_table( index="city", columns="room_type", values="price", aggfunc="median").round(1)
display(median_price_by_city_room)

In [ ]:
# Impact of cancellation policy and instant bookability on price
price_cap = df["price"].quantile(0.99)
price_data = df[df["price"] <= price_cap]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(data=price_data, x="cancellation_policy", y="price", ax=axes[0], palette="Set2")
axes[0].set(title="Price by Cancellation Policy", xlabel="Cancellation Policy", ylabel="Nightly Price (USD)")

sns.boxplot(data=price_data, x="instant_bookable", y="price", ax=axes[1], palette="Set1")
axes[1].set(title="Price by Instant Bookability", xlabel="Instant Bookable", ylabel="Nightly Price (USD)")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numeric variables
corr_cols = ["log_price", "accommodates", "bedrooms", "bathrooms", "beds",
             "number_of_reviews", "review_scores_rating"]
corr_cols = [col for col in corr_cols if col in df.columns]

plt.figure(figsize=(8, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="Blues", cbar=True, square=True)
plt.title("Correlation Heatmap with Target (log_price)")
plt.tight_layout()
plt.show()

### Key EDA Findings
Use the charts and summary tables above to inspect price skew, compare listing prices across room types and cities, and examine how capacity features relate to `log_price`. Treat the plots as descriptive evidence; use the values produced by the current dataset run rather than assuming fixed results.

## 08 - Feature Engineering

### Domain Feature Construction
To capture non-linear domain dynamics, we construct new predictors from raw fields:
* **Capacity & Space Ratios:** `guests_per_bedroom`, `guests_per_bed`, and `bath_per_guest` capture space generosity vs. crowding.
* **Amenity Parsing:** Parses the raw amenity string into an overall `amenity_count` plus individual binary flags for top high-impact amenities (*Air conditioning*, *Pool*, *Kitchen*, *Washer/Dryer*).
* **Geospatial Distance:** `dist_to_centre_km` computes the haversine distance to each respective city center (e.g., Times Square in NYC, The Loop in Chicago).
* **Host Signals:** `host_tenure_years` calculates years of experience from `host_since`, and `host_response_rate` is converted to a numeric percentage.


In [ ]:
def engineer_features(data):
    """Create amenity, capacity, host, review, and location features."""
    df_fe = data.copy()

    # Count amenities and create indicators for popular amenities
    amenities = df_fe["amenities"].fillna("").astype(str)
    df_fe["amenity_count"] = amenities.apply(
        lambda value: len(re.findall(r'"([^"]+)"|([^,{}]+)', value))
    )
    popular_amenities = [
        "Wireless Internet", "Air conditioning", "Kitchen", "Heating",
        "Family/kid friendly", "Essentials", "Hair dryer", "Iron",
        "Smoke detector", "TV", "Washer", "Dryer", "Pool",
    ]
    for amenity in popular_amenities:
        feature_name = "amen_" + re.sub(r"[^a-z0-9]+", "_", amenity.lower()).strip("_")
        df_fe[feature_name] = amenities.str.contains(amenity, case=False).astype(int)

    # Parse host response rates; invalid or missing values remain missing for imputation
    df_fe["host_response_rate"] = pd.to_numeric(
        df_fe["host_response_rate"].astype(str).str.replace("%", "", regex=False),
        errors="coerce",
    )

    # Create capacity features while preserving missing source values
    guests = df_fe["accommodates"].clip(lower=1)
    bedrooms = df_fe["bedrooms"].clip(lower=1)
    beds = df_fe["beds"].clip(lower=1)
    bathrooms = df_fe["bathrooms"]
    df_fe["guests_per_bedroom"] = guests / bedrooms
    df_fe["guests_per_bed"] = guests / beds
    df_fe["bath_per_guest"] = bathrooms / guests
    df_fe["total_rooms"] = df_fe["bedrooms"] + bathrooms

    # Calculate host tenure using the dataset reference date
    reference_date = pd.Timestamp("2017-10-05")
    host_since = pd.to_datetime(df_fe["host_since"], errors="coerce")
    df_fe["host_tenure_years"] = ((reference_date - host_since).dt.days / 365.25).clip(lower=0)

    # Mark review presence, preserving missing review counts
    review_count = pd.to_numeric(df_fe["number_of_reviews"], errors="coerce")
    df_fe["has_reviews"] = np.where(review_count.isna(), np.nan, (review_count > 0).astype(int))

    # Convert known true/false values; unknowns remain missing for pipeline imputation
    binary_columns = ["cleaning_fee", "instant_bookable", "host_identity_verified", "host_has_profile_pic"]
    for column in binary_columns:
        if column in df_fe.columns:
            values = df_fe[column]
            normalized = values.map({True: "t", False: "f"}).fillna(values)
            df_fe[column] = normalized.astype("string").str.strip().str.lower().map({"t": 1, "f": 0})

    # Calculate distance to the city center in kilometers
    city_centres = {
        "NYC": (40.7580, -73.9855), "LA": (34.0522, -118.2437),
        "SF": (37.7749, -122.4194), "DC": (38.9072, -77.0369),
        "Chicago": (41.8781, -87.6298), "Boston": (42.3601, -71.0589),
    }
    def distance_to_centre(row):
        centre = city_centres.get(row["city"])
        latitude, longitude = row["latitude"], row["longitude"]
        if centre is None or pd.isna(latitude) or pd.isna(longitude):
            return np.nan
        centre_lat, centre_lon = centre
        dlat = np.radians(latitude - centre_lat)
        dlon = np.radians(longitude - centre_lon)
        value = (np.sin(dlat / 2) ** 2 + np.cos(np.radians(centre_lat))
                 * np.cos(np.radians(latitude)) * np.sin(dlon / 2) ** 2)
        return 6371.0 * 2 * np.arcsin(np.sqrt(value))

    df_fe["dist_to_centre_km"] = df_fe.apply(distance_to_centre, axis=1)
    return df_fe


df = engineer_features(df)
print(f"Feature engineering complete: {df.shape[0]:,} rows × {df.shape[1]} columns")

In [ ]:
# Preview selected engineered features
preview_cols = [
    "accommodates", "bedrooms", "guests_per_bedroom", "bath_per_guest",
    "amenity_count", "dist_to_centre_km", "host_tenure_years", "has_reviews",
]
display(df[preview_cols].head())

In [ ]:
# Correlation of engineered features with the target
feature_cols = ["amenity_count", "dist_to_centre_km", "guests_per_bedroom","bath_per_guest", "total_rooms", "host_tenure_years", "has_reviews",]
feature_cols = [col for col in feature_cols if col in df.columns]

feature_correlations = (
    df[feature_cols]
    .corrwith(df["log_price"])
    .sort_values()
)

print("Correlation of engineered features with log_price:")
for feature, correlation in feature_correlations.items():
    print(f"{feature:22s}: {correlation:+.3f}")

## 09 - Data Preprocessing (the Pipeline)
Setting up a leak-free scikit-learn `ColumnTransformer` to handle feature-specific transformations:
* **Numerical Features:** Imputed using **median** strategy and standardized using `StandardScaler`.
* **Categorical Features:** Imputed with `"Unknown"` and encoded using `OneHotEncoder(handle_unknown="ignore")`.
* **Binary Features:** Imputed using most frequent values.


In [ ]:
# Define feature groups
numeric_features = ["accommodates", "bathrooms", "bedrooms", "beds", "number_of_reviews","review_scores_rating", "host_response_rate", "latitude", "longitude", "dist_to_centre_km", "guests_per_bedroom", "guests_per_bed",
    "bath_per_guest", "total_rooms", "host_tenure_years", "amenity_count",]
categorical_features = [
    "city", "property_type", "room_type", "bed_type", "cancellation_policy",]
binary_features = [
    "cleaning_fee", "instant_bookable", "host_identity_verified",
    "host_has_profile_pic", "has_reviews",
] + [col for col in df.columns if col.startswith("amen_")]
all_features = numeric_features + categorical_features + binary_features

# Build the preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
        ("bin", SimpleImputer(strategy="most_frequent"), binary_features),
    ],
    remainder="drop",
)

## 10 - Train-Test Split

In [ ]:
# Split features and target into training and test sets
X = df[all_features]
y = df["log_price"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print(f"Training set: {len(X_train):,} samples (80%)")
print(f"Test set: {len(X_test):,} samples (20%)")

# Verify transformed feature names using training data only
preprocessor.fit(X_train)
encoded_features = preprocessor.get_feature_names_out()
print(f"Input features: {X_train.shape[1]}")
print(f"Transformed features: {len(encoded_features)}")
print("First six transformed feature names:")
for name in encoded_features[:6]:
    print(f"- {name}")

In [ ]:
# Compare target distributions in the training and test sets
split_summary = pd.DataFrame({
    "Split": ["Training Set (80%)", "Test Set (20%)"],
    "Count": [len(y_train), len(y_test)],
    "Mean log_price": [y_train.mean(), y_test.mean()],
    "Std log_price": [y_train.std(), y_test.std()],
}).round(3)

display(split_summary)

## 11 - Model Training
Evaluating candidate regression algorithms ranging from simple baselines to regularized linear models and non-linear tree ensembles using **3-Fold Cross-Validation** on `X_train`:
1. **Baseline (Median):** Predicts median training price — minimum benchmark.
2. **Ridge Regression:** L2-regularized linear model for baseline additive effects.
3. **Decision Tree:** Single tree modeling hierarchical decision splits.
4. **Random Forest:** Bagging ensemble averaging randomized decision trees.
5. **Gradient Boosting & HistGradientBoosting:** Sequential boosting algorithms that minimize residual loss; `HistGradientBoosting` uses binned histograms optimized for large datasets.


In [ ]:
# Define regression models
models = {
    "Baseline (median)": DummyRegressor(strategy="median"),
    "Ridge Regression": Ridge(alpha=10.0),
    "Decision Tree": DecisionTreeRegressor(max_depth=12, min_samples_leaf=20, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=16, min_samples_leaf=5, n_jobs=-1, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=200, learning_rate=0.08, min_samples_leaf=30, random_state=42),
}

# Set up cross-validation and result storage
cv = KFold(n_splits=3, shuffle=True, random_state=42)
results = []
fitted_pipelines = {}

# Train and evaluate each model
for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", model),
    ])

    cv_rmse = -cross_val_score(
        pipeline, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error"
    ).mean()

    pipeline.fit(X_train, y_train)
    fitted_pipelines[name] = pipeline
    predictions_log = pipeline.predict(X_test)

    test_rmse_log = np.sqrt(mean_squared_error(y_test, predictions_log))
    test_mae_log = mean_absolute_error(y_test, predictions_log)
    test_r2_log = r2_score(y_test, predictions_log)

    actual_price = np.exp(y_test)
    predicted_price = np.exp(predictions_log)
    test_mae_usd = mean_absolute_error(actual_price, predicted_price)
    median_percentage_error = np.median(
        np.abs((actual_price - predicted_price) / actual_price)
    ) * 100

    results.append({
        "Model": name,
        "CV RMSE (log)": cv_rmse,
        "Test RMSE (log)": test_rmse_log,
        "Test MAE (log)": test_mae_log,
        "Test R² (log)": test_r2_log,
        "Test MAE (USD)": test_mae_usd,
        "Median Percentage Error (%)": median_percentage_error,
    })

df_results = (
    pd.DataFrame(results)
    .sort_values("CV RMSE (log)")
    .reset_index(drop=True)
)

display(df_results.round(4))

## 12 - Model Comparison
Comparing cross-validation RMSE and held-out test performance across all candidate algorithms.


In [ ]:
# Compare cross-validation and test performance
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=df_results, x="Model", y="CV RMSE (log)", ax=axes[0], palette="Blues_r")
axes[0].set(title="Cross-Validation RMSE (lower is better)", xlabel="Model", ylabel="RMSE (log)")
axes[0].tick_params(axis="x", rotation=25)

sns.barplot(data=df_results, x="Model", y="Test R² (log)", ax=axes[1], palette="Greens_r")
axes[1].set(title="Test R² Score (higher is better)", xlabel="Model", ylabel="R² (log)")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

### Model Comparison Insights
Compare candidates using the displayed cross-validation RMSE and held-out test metrics. Use cross-validation to select a model; keep test-set results as the final performance check. The ranking depends on the current data and run.

### Model Architecture Trade-Offs
Ridge Regression provides a regularized linear baseline. A single Decision Tree can capture nonlinear splits but may overfit, while Random Forest and boosting combine trees to model more complex patterns. Use the measured train/test RMSE gap above to assess generalization; this notebook does not record training times.

In [ ]:
# Compare training and test performance
generalization_results = []

for name, pipeline in fitted_pipelines.items():
    train_predictions = pipeline.predict(X_train)
    test_predictions = pipeline.predict(X_test)
    generalization_results.append({
        "Model": name,
        "Train RMSE": np.sqrt(mean_squared_error(y_train, train_predictions)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, test_predictions)),
    })

overfit_summary = pd.DataFrame(generalization_results)
overfit_summary["Generalization Gap"] = overfit_summary["Test RMSE"] - overfit_summary["Train RMSE"]
display(overfit_summary.sort_values("Test RMSE").round(4))

## 13 - Hyperparameter Tuning
Fine-tuning the winning algorithm (`HistGradientBoostingRegressor`) using `RandomizedSearchCV` across learning rates, iterations, leaf nodes, and leaf sample limits with 3-fold cross-validation.


In [ ]:
# Tune HistGradientBoosting with randomized search
param_dist = {
    "regressor__learning_rate": [0.03, 0.05, 0.08, 0.1],
    "regressor__max_iter": [150, 250, 350],
    "regressor__max_leaf_nodes": [31, 63, 127],
    "regressor__min_samples_leaf": [20, 40, 80],
}
base_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", HistGradientBoostingRegressor(random_state=42)),
])
search = RandomizedSearchCV(
    base_pipeline, param_distributions=param_dist, n_iter=6, cv=cv,
    scoring="neg_root_mean_squared_error", random_state=42, n_jobs=-1,
)
search.fit(X_train, y_train)
final_model = search.best_estimator_

print(f"Best hyperparameters: {search.best_params_}")
print(f"Best CV RMSE: {-search.best_score_:.4f}")

## 14 - Model Evaluation
Evaluating the final tuned model on the untouched test set (14,822 listings) across statistical metrics (R², RMSE) and practical business metrics (MAE in dollars, median % error).


In [ ]:
# Compare default and tuned HistGradientBoosting models
default_result = df_results.loc[
    df_results["Model"] == "HistGradientBoosting"
].iloc[0]

tuned_predictions = final_model.predict(X_test)
actual_prices = np.exp(y_test)
tuned_prices = np.exp(tuned_predictions)

tuned_metrics = [
    -search.best_score_,
    np.sqrt(mean_squared_error(y_test, tuned_predictions)),
    r2_score(y_test, tuned_predictions),
    mean_absolute_error(actual_prices, tuned_prices),
]
default_metrics = [
    default_result["CV RMSE (log)"],
    default_result["Test RMSE (log)"],
    default_result["Test R² (log)"],
    default_result["Test MAE (USD)"],
]

tuning_comparison = pd.DataFrame({
    "Metric": ["CV RMSE (log)", "Test RMSE (log)", "Test R² (log)", "Test MAE (USD)"],
    "Default Model": default_metrics,
    "Tuned Model": tuned_metrics,
})
tuning_comparison["Change (Tuned - Default)"] = (
    tuning_comparison["Tuned Model"] - tuning_comparison["Default Model"]
)
display(tuning_comparison.round(4))

In [ ]:
# Diagnostic evaluation plots
final_preds = final_model.predict(X_test)
price_true = np.exp(y_test.to_numpy())
price_pred = np.exp(final_preds)
residuals = y_test.to_numpy() - final_preds

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# Actual vs. predicted log price
ax = axes[0, 0]
ax.scatter(y_test, final_preds, s=5, alpha=0.25, color="#4C72B0")
limits = [min(y_test.min(), final_preds.min()), max(y_test.max(), final_preds.max())]
ax.plot(limits, limits, "r--", label="Perfect prediction")
ax.set(title="Actual vs. Predicted (log price)", xlabel="Actual log(price)", ylabel="Predicted log(price)")
ax.legend()

# Actual vs. predicted price in USD, shown up to the 99th percentile
ax = axes[0, 1]
price_cap = np.quantile(price_true, 0.99)
ax.scatter(price_true, price_pred, s=5, alpha=0.25, color="#55A868")
ax.plot([0, price_cap], [0, price_cap], "r--", label="Perfect prediction")
ax.set(xlim=(0, price_cap), ylim=(0, price_cap), title="Actual vs. Predicted Price (up to 99th percentile)",
       xlabel="Actual nightly price (USD)", ylabel="Predicted nightly price (USD)")
ax.legend()

# Residual distribution
ax = axes[1, 0]
sns.histplot(residuals, bins=70, kde=True, color="#C44E52", ax=ax)
ax.axvline(0, color="black", ls="--")
ax.set(title=f"Residuals (mean = {residuals.mean():.3f}, std = {residuals.std():.3f})",
       xlabel="Residual (actual − predicted log price)", ylabel="Count")

# Residuals vs. predicted log price
ax = axes[1, 1]
ax.scatter(final_preds, residuals, s=5, alpha=0.25, color="#8172B2")
ax.axhline(0, color="black", ls="--")
ax.set(title="Residuals vs. Predicted Log Price", xlabel="Predicted log(price)", ylabel="Residual")

plt.tight_layout()
plt.show()

### Performance Interpretation
Use the evaluation table and diagnostic plots above to interpret the current run. R² and RMSE are calculated on `log_price`; MAE in USD and percentage-error summaries are calculated after converting predictions back to dollars. Exponentiating a log prediction gives a median-style price estimate and does not apply a bias correction for the mean price. Check residual plots for patterns rather than assuming they are normally distributed or unbiased.

In [ ]:
# Model performance by city
eval_df = pd.DataFrame({
    "city": X_test["city"].to_numpy(),
    "actual_price": price_true,
    "predicted_price": price_pred,
})
eval_df["abs_error"] = (eval_df["actual_price"] - eval_df["predicted_price"]).abs()
eval_df["pct_error"] = eval_df["abs_error"] / eval_df["actual_price"] * 100
city_breakdown = eval_df.groupby("city").agg(
    Listings=("actual_price", "count"),
    MAE_USD=("abs_error", "mean"),
    Median_Pct_Error=("pct_error", "median"),
).round(2)
display(city_breakdown)

In [ ]:
# Analyze the share of predictions within selected error margins
actual_prices = np.asarray(price_true)
predicted_prices = np.asarray(price_pred)
percentage_errors = np.abs(predicted_prices - actual_prices) / actual_prices

tolerances = [0.10, 0.20, 0.25, 0.30, 0.50]
error_thresholds = pd.DataFrame({
    "Error Margin": [f"Within ±{tolerance:.0%}" for tolerance in tolerances],
    "Share of Listings": [
        f"{np.mean(percentage_errors <= tolerance):.1%}" for tolerance in tolerances
    ],
    "Practical Meaning": [
        "Highly precise starting price",
        "Solid estimate for common listings",
        "Within acceptable host variance",
        "Reasonable commercial guidance",
        "Broad pricing boundary",
    ],
})
display(error_thresholds)

## 15 - Feature Importance and Model Explainability
Extracting feature importances from the tree ensemble to explain which listing characteristics drive price predictions most strongly.


In [ ]:
# Top 15 feature importances from the Random Forest pipeline
rf_pipeline = fitted_pipelines["Random Forest"]
rf_model = rf_pipeline.named_steps["regressor"]
encoded_features = rf_pipeline.named_steps["preprocessor"].get_feature_names_out()

feature_importances = pd.Series(rf_model.feature_importances_, index=encoded_features)
top_features = feature_importances.nlargest(15).sort_values()

plt.figure(figsize=(10, 6))
top_features.plot(kind="barh", color="#4C72B0")
plt.title("Top 15 Feature Importances (Random Forest)")
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.show()

### Key Value Drivers
The feature importance charts summarize which inputs the fitted Random Forest uses for prediction. Importance indicates predictive contribution in this model, not causal effect. One-hot encoded categories may split the importance of a feature across several encoded columns.

In [ ]:
# Group Random Forest importance by feature category
def get_feature_category(name):
    name = name.split("__", 1)[-1]
    if name in {"accommodates", "bedrooms", "bathrooms", "beds", "guests_per_bedroom",
                "guests_per_bed", "bath_per_guest", "total_rooms"}:
        return "Property Capacity"
    if name.startswith("room_type"):
        return "Room Type"
    if name in {"latitude", "longitude", "dist_to_centre_km"} or name.startswith("city"):
        return "Geographic Location"
    if name.startswith(("property_type", "bed_type", "cancellation_policy",
                        "instant_bookable", "cleaning_fee")):
        return "Property Type & Booking Rules"
    if name.startswith(("amen_", "amenity_count", "host_")):
        return "Amenities & Host Signals"
    return "Reviews & Ratings"

grouped_importance = feature_importances.rename_axis("Feature").reset_index(name="Importance")
grouped_importance["Feature Category"] = grouped_importance["Feature"].map(get_feature_category)
grouped_importance = (
    grouped_importance.groupby("Feature Category", as_index=False)["Importance"].sum()
)
grouped_importance["Relative Contribution (%)"] = (
    grouped_importance["Importance"] / grouped_importance["Importance"].sum() * 100
).round(2)
grouped_importance = grouped_importance.sort_values("Relative Contribution (%)", ascending=False)
display(grouped_importance[["Feature Category", "Relative Contribution (%)"]])

## 16 - User Input Prediction System
Creating a standalone prediction function `predict_price` that accepts a raw listing dictionary, automatically applies feature engineering and preprocessing, and returns the predicted nightly price in USD.


In [ ]:
def predict_price(listing_data):
    """Predict an Airbnb listing's nightly price in USD."""
    required_fields = ["amenities", "host_response_rate", "accommodates", "bedrooms", "beds",
                       "bathrooms", "host_since", "number_of_reviews", "city", "latitude", "longitude"]
    missing_fields = [field for field in required_fields if field not in listing_data]
    if missing_fields:
        raise ValueError(f"Missing required listing fields: {', '.join(missing_fields)}")
    input_df = engineer_features(pd.DataFrame([listing_data]))
    for column in all_features:
        if column not in input_df.columns:
            input_df[column] = np.nan
    log_price_prediction = final_model.predict(input_df[all_features])[0]
    return round(float(np.exp(log_price_prediction)), 2)

# Example 1: Entire apartment in NYC
sample_1 = {
    "city": "NYC", "room_type": "Entire home/apt", "property_type": "Apartment",
    "accommodates": 4, "bedrooms": 2, "bathrooms": 1, "beds": 2,
    "bed_type": "Real Bed", "cancellation_policy": "strict",
    "cleaning_fee": True, "instant_bookable": True,
    "host_identity_verified": True, "host_has_profile_pic": True,
    "number_of_reviews": 25, "review_scores_rating": 95,
    "host_response_rate": "100%", "host_since": "2015-06-01",
    "latitude": 40.7580, "longitude": -73.9855,
    "amenities": '{"Wireless Internet","Air conditioning",Kitchen,Heating,TV}',
}

# Example 2: Private room in Los Angeles
sample_2 = {
    "city": "LA", "room_type": "Private room", "property_type": "House",
    "accommodates": 2, "bedrooms": 1, "bathrooms": 1, "beds": 1,
    "bed_type": "Real Bed", "cancellation_policy": "flexible",
    "cleaning_fee": False, "instant_bookable": False,
    "host_identity_verified": True, "host_has_profile_pic": True,
    "number_of_reviews": 5, "review_scores_rating": 88,
    "host_response_rate": "90%", "host_since": "2016-01-15",
    "latitude": 34.0522, "longitude": -118.2437,
    "amenities": '{"Wireless Internet",Essentials}',
}
print(f"NYC entire apartment: ${predict_price(sample_1):,.2f} per night")
print(f"LA private room: ${predict_price(sample_2):,.2f} per night")

In [ ]:
# What-If Sensitivity Analysis: Changing Room Type for the NYC listing
sample_entire = sample_1.copy()
sample_private = sample_1.copy()
sample_private["room_type"] = "Private room"

price_entire = predict_price(sample_entire)
price_private = predict_price(sample_private)

print(f"Original Listing (Entire Apt, NYC):   ${price_entire:.2f} / night")
print(f"Modified Listing (Private Room, NYC): ${price_private:.2f} / night")
print(f"Price Difference:                     -${price_entire - price_private:.2f} ({(1 - price_private/price_entire)*100:.1f}% reduction)")

In [ ]:
# Compare market scenarios and estimate monthly gross revenue
occupancy_rate = 0.70
nights_per_month = 30
booked_nights = nights_per_month * occupancy_rate

scenarios = {
    "NYC 2-Bed Entire Apartment": sample_1,
    "LA 1-Bed Private Room": sample_2,
    "Chicago Downtown Studio": {
        "city": "Chicago", "room_type": "Entire home/apt", "property_type": "Apartment",
        "accommodates": 2, "bedrooms": 0, "bathrooms": 1, "beds": 1,
        "bed_type": "Real Bed", "cancellation_policy": "flexible",
        "cleaning_fee": True, "instant_bookable": True,
        "host_identity_verified": True, "host_has_profile_pic": True,
        "number_of_reviews": 12, "review_scores_rating": 92,
        "host_response_rate": "95%", "host_since": "2016-08-01",
        "latitude": 41.8781, "longitude": -87.6298,
        "amenities": '{"Wireless Internet",Kitchen,Heating}',
    },
}

revenue_rows = []
for scenario, listing in scenarios.items():
    nightly_price = predict_price(listing)
    revenue_rows.append({
        "Listing Scenario": scenario,
        "Predicted Nightly Price (USD)": f"${nightly_price:,.2f}",
        "Suggested Nightly Range (±15%)": f"${nightly_price * 0.85:,.2f}–${nightly_price * 1.15:,.2f}",
        f"Estimated Monthly Gross ({booked_nights:.0f} nights)": f"${nightly_price * booked_nights:,.2f}",
    })

revenue_table = pd.DataFrame(revenue_rows)
display(revenue_table)

## 17 - Model Saving and Loading
Serializing the final trained pipeline to disk with `joblib` and verifying that the saved model can be reloaded and used independently for inference.


In [ ]:
# Save and reload the tuned model pipeline
model_path = "airbnb_price_prediction_model.pkl"
joblib.dump(final_model, model_path)
print(f"Model pipeline saved to {model_path}")
reloaded_model = joblib.load(model_path)
reloaded_predictions = reloaded_model.predict(X_test.iloc[:5])
print(f"Reload verified: generated {len(reloaded_predictions)} sample predictions")

In [ ]:
# Save metadata using metrics from the current run
metadata = {
    "project_name": "Airbnb Nightly Price Prediction",
    "model_type": f"{type(final_model.named_steps['regressor']).__name__} pipeline",
    "training_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "r2_score": float(r2_score(y_test, final_preds)),
    "rmse_log": float(np.sqrt(mean_squared_error(y_test, final_preds))),
    "mae_usd": float(mean_absolute_error(price_true, price_pred)),
    "median_pct_error": float(np.median(np.abs(price_pred - price_true) / price_true) * 100),
    "target_column": "log_price",
    "supported_cities": ["NYC", "LA", "SF", "DC", "Chicago", "Boston"],
    "feature_count": int(len(all_features)),
}
with open("model_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)
print("Model metadata saved to model_metadata.json")

## 18 - Project Achievements

### Project Summary
This notebook covers data inspection and cleaning, feature engineering, leak-free preprocessing, model comparison, hyperparameter tuning, evaluation, and single-listing price prediction. Final metrics are calculated from the held-out test set.

## 19 - Conclusion
### Practical Use
Treat each prediction as a data-based starting price estimate. Compare it with similar local listings and current market conditions before setting a nightly rate.

### Limitations and Future Work
1. **Seasonality:** The dataset is a cross-sectional snapshot and does not model seasonal demand or holidays.
2. **Listing text:** NLP on titles and descriptions could capture qualitative factors such as views and interior quality.
3. **Local context:** Transit access, walkability, and neighborhood-level data could improve spatial estimates.

## 20 - Final Test Evaluation

The final cell reports the tuned model's regression metrics and displays a preview of test-set predictions beside actual prices. The full comparison is stored in `predicted_vs_actual`.

In [ ]:
# Final test metrics and actual-versus-predicted prices
final_predictions_log = final_model.predict(X_test)
actual_prices_usd = np.exp(y_test.to_numpy())
predicted_prices_usd = np.exp(final_predictions_log)
absolute_errors_usd = np.abs(actual_prices_usd - predicted_prices_usd)
percentage_errors = absolute_errors_usd / actual_prices_usd * 100

metric_rows = [
    ("Best cross-validation RMSE (log price)", -search.best_score_, "log-price units"),
    ("Test R² score (log price)", r2_score(y_test, final_predictions_log), "score"),
    ("Test RMSE (log price)", np.sqrt(mean_squared_error(y_test, final_predictions_log)), "log-price units"),
    ("Test MAE (log price)", mean_absolute_error(y_test, final_predictions_log), "log-price units"),
    ("Test MAE", mean_absolute_error(actual_prices_usd, predicted_prices_usd), "USD"),
    ("Median absolute percentage error", np.median(percentage_errors), "%"),
    ("Predictions within ±10%", np.mean(percentage_errors <= 10) * 100, "% of listings"),
    ("Predictions within ±20%", np.mean(percentage_errors <= 20) * 100, "% of listings"),
    ("Predictions within ±25%", np.mean(percentage_errors <= 25) * 100, "% of listings"),
    ("Predictions within ±30%", np.mean(percentage_errors <= 30) * 100, "% of listings"),
    ("Predictions within ±50%", np.mean(percentage_errors <= 50) * 100, "% of listings"),
]
final_metrics = pd.DataFrame(metric_rows, columns=["Metric", "Value", "Unit"])
final_metrics["Value"] = final_metrics["Value"].round(4)
print(f"Final evaluation on {len(y_test):,} held-out listings")
display(final_metrics)

predicted_vs_actual = pd.DataFrame({
    "Actual log_price": y_test.to_numpy(),
    "Predicted log_price": final_predictions_log,
    "Actual price (USD)": actual_prices_usd,
    "Predicted price (USD)": predicted_prices_usd,
    "Absolute error (USD)": absolute_errors_usd,
    "Absolute percentage error (%)": percentage_errors,
}, index=X_test.index)
print("Actual vs. predicted prices (first 20 test listings)")
display(predicted_vs_actual.head(20).round(2))